In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf

# 2. Load the 'cats_vs_dogs' dataset
(train_dataset, test_dataset), info = tfds.load(
    'cats_vs_dogs',
    with_info=True,
    as_supervised=True,
    split=['train[:80%]', 'train[80%:]']
)


IMG_WIDTH = 150
IMG_HEIGHT = 150

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    image = tf.cast(image, tf.float32) / 255.0 # Normalize to [0,1]
    return image, label


train_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


BUFFER_SIZE = info.splits['train'].num_examples
BATCH_SIZE = 32

train_dataset = train_dataset.cache()
train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)


test_dataset = test_dataset.cache()
test_dataset = test_dataset.batch(BATCH_SIZE)


train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

print("Dataset preparation complete. \nTraining dataset and Test dataset are ready.")
print(f"Number of training batches: {tf.data.experimental.cardinality(train_dataset).numpy()}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset).numpy()}")


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.60EMLH_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.
Dataset preparation complete. 
Training dataset and Test dataset are ready.
Number of training batches: 582
Number of test batches: 146


In [2]:
import tensorflow_datasets as tfds
import tensorflow as tf

# 2. Load the 'cats_vs_dogs' dataset
(train_dataset, test_dataset), info = tfds.load(
    'cats_vs_dogs',
    with_info=True,
    as_supervised=True,
    split=['train[:80%]', 'train[80%:]']
)


IMG_WIDTH = 150
IMG_HEIGHT = 150

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    image = tf.cast(image, tf.float32) / 255.0 # Normalize to [0,1]
    return image, label


train_dataset = train_dataset.map(preprocess)
test_dataset = test_dataset.map(preprocess)


BUFFER_SIZE = info.splits['train'].num_examples
BATCH_SIZE = 32

train_dataset = train_dataset.cache()
train_dataset = train_dataset.shuffle(BUFFER_SIZE)
train_dataset = train_dataset.batch(BATCH_SIZE)


test_dataset = test_dataset.cache()
test_dataset = test_dataset.batch(BATCH_SIZE)


train_dataset = train_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

print("Dataset preparation complete. \nTraining dataset and Test dataset are ready.")
print(f"Number of training batches: {tf.data.experimental.cardinality(train_dataset).numpy()}")
print(f"Number of test batches: {tf.data.experimental.cardinality(test_dataset).numpy()}")


Dataset preparation complete. 
Training dataset and Test dataset are ready.
Number of training batches: 582
Number of test batches: 146


In [3]:
from tensorflow.keras import layers

# 1. Create a data augmentation model
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(height_factor=0.2, width_factor=0.2)
])

# 2. Define a helper function to apply augmentation to the training dataset
def apply_data_augmentation(image, label):
    image = data_augmentation(image) # Apply augmentation to the image
    return image, label

# 3. Apply this augmentation function to the training dataset
# Only apply augmentation to the training dataset
train_dataset_augmented = train_dataset.map(apply_data_augmentation, num_parallel_calls=tf.data.AUTOTUNE)

print("Data augmentation applied to the training dataset.")

Data augmentation applied to the training dataset.


In [4]:
from tensorflow.keras import models, layers, Input

# 2. Define the input shape for the model
IMG_SHAPE = (IMG_WIDTH, IMG_HEIGHT, 3) # RGB images have 3 channels

# 3. Build the CNN model using the Functional API
# Using Input layer for clarity and flexibility
inputs = Input(shape=IMG_SHAPE)

# First convolutional block
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
x = layers.MaxPooling2D((2, 2))(x)

# Second convolutional block
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Third convolutional block
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Flatten the output to feed into dense layers
x = layers.Flatten()(x)

# Fully connected layers
x = layers.Dense(128, activation='relu')(x)

# Output layer for binary classification
outputs = layers.Dense(1, activation='sigmoid')(x)

# 4. Create the tf.keras.Model instance
model = models.Model(inputs=inputs, outputs=outputs)

# 5. Print a summary of the model
print("Model Summary:")
model.summary()


Model Summary:


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     5,308,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,401,921 (20.61 MB)

 Trainable params: 5,401,921 (20.61 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compilation complete.")

Model compilation complete.


In [6]:
EPOCHS = 10

history = model.fit(
    train_dataset_augmented,
    epochs=EPOCHS,
    validation_data=test_dataset
)

print("Model training complete. Training history stored.")

# Print final accuracy and loss
final_accuracy = history.history['accuracy'][-1]
final_val_accuracy = history.history['val_accuracy'][-1]
final_loss = history.history['loss'][-1]
final_val_loss = history.history['val_loss'][-1]

print(f"Final Training Accuracy: {final_accuracy:.4f}")
print(f"Final Validation Accuracy: {final_val_accuracy:.4f}")
print(f"Final Training Loss: {final_loss:.4f}")
print(f"Final Validation Loss: {final_val_loss:.4f}")

Epoch 1/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 123s 163ms/step - accuracy: 0.5605 - loss: 0.6866 - val_accuracy: 0.6752 - val_loss: 0.6009
Epoch 2/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 106s 146ms/step - accuracy: 0.6737 - loss: 0.6043 - val_accuracy: 0.6518 - val_loss: 0.6186
Epoch 3/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 86s 147ms/step - accuracy: 0.6861 - loss: 0.5827 - val_accuracy: 0.7113 - val_loss: 0.5745
Epoch 4/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 141s 146ms/step - accuracy: 0.7109 - loss: 0.5614 - val_accuracy: 0.7382 - val_loss: 0.5197
Epoch 5/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 143s 148ms/step - accuracy: 0.7266 - loss: 0.5426 - val_accuracy: 0.7304 - val_loss: 0.5324
Epoch 6/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 140s 144ms/step - accuracy: 0.7302 - loss: 0.5335 - val_accuracy: 0.7577 - val_loss: 0.4944
Epoch 7/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 85s 146ms/step - accuracy: 0.7471 - loss: 0.5147 - val_accuracy: 0.7558 - val_loss: 0.5012
Epoch 8/10
582/582 ━━━━━━━━━━━━━━━━━━━━ 143s 147ms/step - accuracy: 0.7554 - l